# fase 3a do churn intelligence — indexação e construção do vectorstore

na fase 1a a gente trabalhou com o dataset estruturado do kaggle, realizando a EDA que embasou as decisões da modelagem; na fase 1b a gente coletou 242 reclamações reais da Claro no Reclame Aqui e aplicou análise de sentimento via pysentimiento, gerando o arquivo `reclamacoes_com_sentimento.csv`; já na fase 2, o modelo XGBoost foi treinado, os clientes com maior risco de churn foram identificados e usamos SHAP para explicar os motivos por trás de cada previsão.

o modelo sabe "quem" vai cancelar e "por que", mas não sabe "o que fazer" com isso. é aqui que o RAG (Retrieval-Augmented Generation) entra.

a lógica é: se o SHAP identificou que o motivo principal de churn de um cliente é "suporte técnico ruim", o sistema vai buscar nas reclamações reais de outros clientes e no FAQ da Claro os trechos mais relevantes sobre esse tema, e passar esse contexto para um LLM gerar uma recomendação de ação para o time de retenção. o LLM não inventa, ele responde baseado em documentação real.

mas é importante relembrar que o dataset do kaggle e as reclamações do Reclame Aqui são de empresas diferentes. a intenção original era integrar as reclamações como feature de sentimento no modelo preditivo, mas a API pública do reclame aqui não retornou as colunas necessárias com qualidade suficiente (score, status de resolução e nota do cliente vieram vazios ou sem variabilidade). qualquer linkagem seria puramente arbitrária. então, as reclamações foram redirecionadas para o RAG, onde o texto real tem papel direto e natural: contextualizar os motivos de churn identificados pelo SHAP com experiências reais de clientes insatisfeitos.

esse notebook cobre a primeira etapa do pipeline de RAG: preparar e indexar as
duas fontes de conhecimento. o output é um vectorstore ChromaDB persistido
no drive, que vai ser consultado nas próximas etapas

## fontes de conhecimento

duas fontes alimentam o vectorstore:

- reclamações do reclame aqui (`reclamacoes_com_sentimento.csv`): que é o texto bruto das 242 reclamações reais de clientes da Claro, com texto limpo e classificação de sentimento (NEG/NEU) gerada pelo pysentimiento na fase 1b. os metadados de sentimento são preservados como campos no ChromaDB, permitindo filtros e enriquecendo o contexto que o LLM recebe na geração.

- FAQ da claro (`claro_faq.md`): que são artigos reais do centro de ajuda da
Claro (claro.com.br/faq), estruturados em markdown por categoria. a seleção desses arquivos foi guiada pelos motivos SHAP dominantes da fase 2: suporte técnico, cobrança, equipamento e atendimento. vale deixar registrado que o FAQ público da Claro tem cobertura limitada em tópicos de cancelamento. essa é uma lacuna comum em empresas do setor, e isso vai aparecer na fase de avaliação do RAG.

In [4]:
!pip install -q langchain langchain-community langchain-huggingface langchain-core langchain-text-splitters chromadb sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.41.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.41.1 which is incompatible.


In [2]:
# versões utilizadas neste notebook
# pra reproduzir o ambiente, descomente e execute:

# !pip install langchain==1.2.15 langchain-community==0.4.1 langchain-core==1.3.3 \
#   langchain-text-splitters==1.1.2 langchain-huggingface==1.2.2 \
#   chromadb==1.5.9 sentence-transformers==5.4.1

In [5]:
import os
import requests
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import drive

In [6]:
# vamo definir a variável de configuração compartilhada entre todas as etapas dessa fase 3
# trocar o modelo invalida o vectorstore, então não podemos alterar esse valor depois da indexação
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
#-> ele roda localmente sem custo de API e tem boa performance em tarefas de similaridade semântica

# vamo configurar os caminhos
DRIVE_BASE = "/content/drive/MyDrive/churn-intelligence"
CHROMA_PATH = f"{DRIVE_BASE}/vectorstore/chroma_db"

# vamo incluir as urls raw do github
URL_RECLAMACOES = "https://raw.githubusercontent.com/meurii/churn-intelligence/refs/heads/main/data/reclamacoes_com_sentimento.csv"
URL_FAQ = "https://raw.githubusercontent.com/meurii/churn-intelligence/refs/heads/main/claro_faq.md"

# vamo definir os parâmetros de chunking
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50 #-> vai evitar cortes abruptos de contexto entre chunks adjacentes

In [7]:
# vamo montar o drive e criar a pasta de destino

drive.mount("/content/drive")
os.makedirs(CHROMA_PATH, exist_ok=True)

Mounted at /content/drive


In [8]:
# vamo carregar as reclamações

response = requests.get(URL_RECLAMACOES)
with open("reclamacoes_com_sentimento.csv", "wb") as f:
    f.write(response.content)

df_reclamacoes = pd.read_csv("reclamacoes_com_sentimento.csv")
print(f"reclamações carregadas: {len(df_reclamacoes)} linhas")
print(df_reclamacoes.columns.tolist())

reclamações carregadas: 242 linhas
['created', 'userCity', 'userState', 'description', 'title', 'sentimento']


In [9]:
# vamo carregar o FAQ tbm

response = requests.get(URL_FAQ)
faq_texto = response.text
print(f"FAQ carregado: {len(faq_texto)} caracteres")
print(faq_texto[:500])

FAQ carregado: 9984 caracteres
# FAQ Claro — Base de Conhecimento para RAG
# Fonte: https://www.claro.com.br/faq
# Coletado em: maio/2026

---

## CATEGORIA: Suporte Técnico

### Como agendar uma visita técnica?
Para agendar uma visita técnica Claro, acesse o Minha Claro Residencial e faça login com seus dados de CPF/CNPJ ou usuário e senha. Vá nos menus > Atendimento > Canais de atendimento > Fale com a Claro > Suporte técnico. Por lá, você pode agendar uma visita técnica.

As opções de suporte técnico também incluem visitas


vamo usar o ChromaDB como banco vetorial pq ele roda localmente sem servidor externo, além de ser gratuito e persistir os dados em disco (o que vai permitir carregar o vectorstore nas fases seguintes sem necessidade de reindexar tudo). ao contrário de outros bancos que exigem conta e API key, o ChromaDB funciona direto aqui no Colab só com uma linha de configuração. além disso, ele tbm permite filtros por metadados de busca, o que vai ser muito útil pra comparar a qualidade dos chunks recuperados por origem nas próximas etapas dessa fase 3.

então, antes da gente indexar qualquer coisa, vamo entender o formato que o pipeline espera.

o LangChain trabalha com um objeto chamado `Document` que tem dois campos:
- `page_content`, que é o texto em si e
- `metadata`, que é um dicionário com qualquer informação extra que a gente queira carregar junto.

esse é o formato que o splitter, o ChromaDB e o retriever esperam receber, e é, basicamente, a unidade básica de dados que circula pelo pipeline inteiro.

In [10]:
# vamo converter os arquivos carregados pra Documents do LangChain

# cada linha das reclamações vai virar um Document, com metadados de sentimento preservados
docs_reclamacoes = []
for _, row in df_reclamacoes.iterrows():
    docs_reclamacoes.append(Document(
        page_content=str(row["description"]),
        #-> vamo usar o description (e não o title) como page content
        #-> o conteúdo da reclamação é o que carrega informação útil, tendo mais valor semântico na busca do que o title
        metadata={
            "origin": "reclamacao",
            "sentimento": str(row["sentimento"]),
            #-> vamo guardar só "sentimento" dos metadados de "reclamações"
            #-> os outros não agregam valor na busca semântica nem na recomendação gerada
        }
    ))

# o FAQ vira um Document único com metadado de origem
# o splitter vai cuidar dos cortes por artigo no chunking
docs_faq = [Document(
    page_content=faq_texto,
    metadata={"origin": "faq"}
)]

print(f"docs de reclamações: {len(docs_reclamacoes)}")
print(f"docs de faq: {len(docs_faq)}")

docs de reclamações: 242
docs de faq: 1


o LLM tem um limite de tokens que consegue processar de uma vez, por isso não dá pra jogar o FAQ inteiro no prompt. além disso, chunks menores tendem a ser mais precisos na busca semântica. então, se a gente for buscar "problema com fatura", um chunk de 400 caracteres sobre cobrança vai ser muito mais relevante do que um documento inteiro que fala de cobrança em um parágrafo e de suporte técnico em outros cinco. sendo assim, como próximo passo, vamo fazer esse processo de chunking.

vamo usar o RecursiveCharacterTextSplitter, que vai tentar cortar o texto usando os separadores na ordem que a gente definir, respeitando a hierarquia de headers markdown, fazendo os artigos FAQ tenderem a virar chunks naturais por seção. a palavra "Recursive" no nome vem exatamente disso: ele aplica os separadores recursivamente do mais estrutural pro mais granular.

In [11]:
# vamo começar instanciando o splitter com as regras de corte que a gente já tinha definido
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
    #-> pra garantir que ele prefira cortar em fronteiras naturais do texto antes de cortar no meio de uma frase
)

# vamo aplicar nas duas fontes separadamente
chunks_reclamacoes = splitter.split_documents(docs_reclamacoes)
chunks_faq = splitter.split_documents(docs_faq)
# e depois juntar tudo em uma lista única pra indexar no ChromaDB
chunks_totais = chunks_reclamacoes + chunks_faq

print(f"chunks de reclamações: {len(chunks_reclamacoes)}")
print(f"chunks de faq:         {len(chunks_faq)}")
print(f"total de chunks:       {len(chunks_totais)}")

chunks de reclamações: 242
chunks de faq:         40
total de chunks:       282


a gente chega até aqui com 282 chunks de texto, entre reclamações e artigos do FAQ, prontos pra ser consultados. acontece que o ChromaDB não busca por texto, e sim por proximidade matemática entre vetores, então, cada chunk precisa ser convertido pra esse formato numérico.

é nesse ponto que entram os embeddings, que são a representação numérica de um texto. o modelo lê uma string de palavras e devolve um vetor de 384 números. o que acontece, basicamente, é que textos com significado semelhante geram vetores próximos no espaço matemático. então, quando a gente buscar por "problema com fatura", vai conseguir encontrar um chunk que fala de "cobrança
indevida" ou "valor incorreto na conta", pq as palavras são diferentes, mas os
vetores ficam próximos.

In [12]:
# vamo  gerar os embeddings e indexar no ChromaDB

print(f"carregando modelo de embedding: {EMBEDDING_MODEL}")
# começa carregando o modelo de embedding na memória
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# depois o Chroma.from_documents passa cada chunk pelo modelo pra gerar o vetor correspondente,
# armazena o par (texto + vetor + metadados) no ChromaDB, e persiste tudo no caminho do drive que a gente configurou
print("indexando chunks no ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks_totais,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
)

print(f"\nvectorstore salvo em: {CHROMA_PATH}")
print(f"total de chunks indexados: {vectorstore._collection.count()}")
print(f"  - reclamações: {len(chunks_reclamacoes)}")
print(f"  - faq:         {len(chunks_faq)}")

carregando modelo de embedding: sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

indexando chunks no ChromaDB...

vectorstore salvo em: /content/drive/MyDrive/churn-intelligence/vectorstore/chroma_db
total de chunks indexados: 564
  - reclamações: 242
  - faq:         40


In [13]:
# vamo incluir um teste de sanidade

# aqi nesse notebook a gente só constrói a base que o LLM vai consultar
# então, o objetivo aqui é confirmar que o vectorstore foi criado corretamente e que a busca semântica está funcionando

query_teste = "problema com sinal de internet"
resultados = vectorstore.similarity_search(query_teste, k=3)

print(f"query de teste: '{query_teste}'")
print(f"chunks recuperados: {len(resultados)}\n")
for i, doc in enumerate(resultados):
    print(f"[{i+1}] origin: {doc.metadata['origin']}")
    print(f"     {doc.page_content[:150]}...")
    print()

query de teste: 'problema com sinal de internet'
chunks recuperados: 3

[1] origin: reclamacao
     segundo a claro, an internet residencial está com problemas na região do brás. a internet parou de funcionar desde o dia 02 04 dev...

[2] origin: reclamacao
     segundo a claro, an internet residencial está com problemas na região do brás. a internet parou de funcionar desde o dia 02 04 dev...

[3] origin: reclamacao
     de novo vindo fazer reclamação sobre a mesma internet e com o mesmo problema. não respondem e não resolvem. quando a internet não...

